# Deep dive 6/13: `daily_exchange_rates`

First REFERENCE-kind table (not a dimension or fact) -- and the first table built with a hand-written function (`build_fx_rates_table` in `silver.py`) instead of a `table_specs.py` `ColumnSpec` list. Two things make it different from every table so far:

1. **The Silver table is renamed and has renamed columns.** `bronze.daily_exchange_rates` -> `silver.dim_fx_rates`; `date` -> `rate_date`, `source` -> `rate_source`. Every other column keeps its name.
2. **The primary key is composite**: `(date, source_currency, target_currency)`, not a single ID column -- `contracts.py` already reflects this via `key_fields=("date", "source_currency", "target_currency")`.
3. **A derived view sits on top of it**: `silver.v_fx_latest_to_usd` (latest USD rate per source currency), used by `products`' currency-conversion logic since a snapshot dimension has no natural transaction date to join on. Section 9 checks the view too, since a bug there would silently corrupt every USD-converted `products`/other-table amount downstream.

Dictionary declares 3,000 rows, Source: Reference, Partition: daily. No FK relationships point at this table (it's joined by `date`/currency, not by ID) -- confirmed in the FK Relationships section of the dictionary, `daily_exchange_rates` doesn't appear as a parent anywhere.

Run top to bottom. Checklist near the end; free cells after that.

In [1]:
import duckdb
import pandas as pd

pd.set_option("display.max_rows", 1000)
pd.set_option("display.min_rows", 1000)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)
pd.set_option("display.max_colwidth", 100)

con = duckdb.connect("data/latam_bank.duckdb")

def q(sql, params=None):
    """Run a query and return a pandas DataFrame."""
    return con.execute(sql, params or []).df()

def q_full(sql, params=None):
    """Like q(), but shows every cell's full text with no truncation."""
    with pd.option_context("display.max_colwidth", None):
        return q(sql, params)

## 0. A look at the table -- sample rows, Bronze and Silver side by side

In [2]:
print("Bronze sample:")
display(q_full("SELECT * FROM bronze.daily_exchange_rates ORDER BY date, source_currency, target_currency LIMIT 20"))

Bronze sample:


,date,source_currency,target_currency,exchange_rate,buy_rate,sell_rate,source,_source_file,_ingested_at,_source_table
0,2023-06-17,ARS,COP,11.533592,11.414077,11.653106,Internal,s3://factored-datathon-2026-s3-157725502942-us-east-2-an/data/daily_exchange_rates.csv,2026-09-28 19:22:58.959745-03:00,daily_exchange_rates
1,2023-06-17,ARS,MXN,0.049247,0.048703,0.04979,Central Bank,s3://factored-datathon-2026-s3-157725502942-us-east-2-an/data/daily_exchange_rates.csv,2026-09-28 19:22:58.959745-03:00,daily_exchange_rates
2,2023-06-17,ARS,USD,0.002877,0.002837,0.002917,Central Bank,s3://factored-datathon-2026-s3-157725502942-us-east-2-an/data/daily_exchange_rates.csv,2026-09-28 19:22:58.959745-03:00,daily_exchange_rates
3,2023-06-17,COP,ARS,0.086108,0.08535,0.086865,Reuters,s3://factored-datathon-2026-s3-157725502942-us-east-2-an/data/daily_exchange_rates.csv,2026-09-28 19:22:58.959745-03:00,daily_exchange_rates
4,2023-06-17,COP,MXN,0.004223,0.004195,0.004251,Reuters,s3://factored-datathon-2026-s3-157725502942-us-east-2-an/data/daily_exchange_rates.csv,2026-09-28 19:22:58.959745-03:00,daily_exchange_rates
5,2023-06-17,COP,USD,0.000247,0.000246,0.000249,Central Bank,s3://factored-datathon-2026-s3-157725502942-us-east-2-an/data/daily_exchange_rates.csv,2026-09-28 19:22:58.959745-03:00,daily_exchange_rates
6,2023-06-17,MXN,ARS,20.801371,20.664153,20.938589,Internal,s3://factored-datathon-2026-s3-157725502942-us-east-2-an/data/daily_exchange_rates.csv,2026-09-28 19:22:58.959745-03:00,daily_exchange_rates
7,2023-06-17,MXN,COP,232.662971,230.128653,235.197288,Central Bank,s3://factored-datathon-2026-s3-157725502942-us-east-2-an/data/daily_exchange_rates.csv,2026-09-28 19:22:58.959745-03:00,daily_exchange_rates
8,2023-06-17,MXN,USD,0.059152,0.058841,0.059462,Reuters,s3://factored-datathon-2026-s3-157725502942-us-east-2-an/data/daily_exchange_rates.csv,2026-09-28 19:22:58.959745-03:00,daily_exchange_rates
9,2023-06-17,USD,ARS,350.857431,346.590936,355.123926,Internal,s3://factored-datathon-2026-s3-157725502942-us-east-2-an/data/daily_exchange_rates.csv,2026-09-28 19:22:58.959745-03:00,daily_exchange_rates


In [3]:
print("Silver sample (note the renamed columns: date -> rate_date, source -> rate_source):")
display(q_full("SELECT * FROM silver.dim_fx_rates ORDER BY rate_date, source_currency, target_currency LIMIT 20"))

Silver sample (note the renamed columns: date -> rate_date, source -> rate_source):


,rate_date,source_currency,target_currency,exchange_rate,buy_rate,sell_rate,rate_source
0,2023-06-17,ARS,COP,11.533592,11.414077,11.653106,Internal
1,2023-06-17,ARS,MXN,0.049247,0.048703,0.049790,Central Bank
2,2023-06-17,ARS,USD,0.002877,0.002837,0.002917,Central Bank
3,2023-06-17,COP,ARS,0.086108,0.085350,0.086865,Reuters
4,2023-06-17,COP,MXN,0.004223,0.004195,0.004251,Reuters
5,2023-06-17,COP,USD,0.000247,0.000246,0.000249,Central Bank
6,2023-06-17,MXN,ARS,20.801371,20.664153,20.938589,Internal
7,2023-06-17,MXN,COP,232.662971,230.128653,235.197288,Central Bank
8,2023-06-17,MXN,USD,0.059152,0.058841,0.059462,Reuters
9,2023-06-17,USD,ARS,350.857431,346.590936,355.123926,Internal


## 1. Column inventory -- names AND types, Bronze vs. Silver (accounting for the rename)

In [4]:
# silver.py's rename map: bronze.date -> silver.rate_date, bronze.source -> silver.rate_source.
# Everything else keeps its name.
expected_silver_columns = [
    "rate_date", "source_currency", "target_currency", "exchange_rate", "buy_rate", "sell_rate", "rate_source",
]

actual_silver_columns = q("""
    SELECT column_name FROM information_schema.columns
    WHERE table_schema='silver' AND table_name='dim_fx_rates'
    ORDER BY ordinal_position
""")["column_name"].tolist()

missing = [c for c in expected_silver_columns if c not in actual_silver_columns]
extra = [c for c in actual_silver_columns if c not in expected_silver_columns]
print("Missing from silver.dim_fx_rates:", missing or "none")
print("Extra in silver.dim_fx_rates:", extra or "none")
print()
print("Column order as built:", actual_silver_columns)

Missing from silver.dim_fx_rates: none
Extra in silver.dim_fx_rates: none

Column order as built: ['rate_date', 'source_currency', 'target_currency', 'exchange_rate', 'buy_rate', 'sell_rate', 'rate_source']


In [5]:
print("Bronze columns + types:")
display(q("""
    SELECT column_name, data_type FROM information_schema.columns
    WHERE table_schema='bronze' AND table_name='daily_exchange_rates' ORDER BY ordinal_position
"""))

print("Silver columns + types:")
display(q("""
    SELECT column_name, data_type FROM information_schema.columns
    WHERE table_schema='silver' AND table_name='dim_fx_rates' ORDER BY ordinal_position
"""))

Bronze columns + types:


,column_name,data_type
0,date,VARCHAR
1,source_currency,VARCHAR
2,target_currency,VARCHAR
3,exchange_rate,VARCHAR
4,buy_rate,VARCHAR
5,sell_rate,VARCHAR
6,source,VARCHAR
7,_source_file,VARCHAR
8,_ingested_at,TIMESTAMP WITH TIME ZONE
9,_source_table,VARCHAR


Silver columns + types:


,column_name,data_type
0,rate_date,DATE
1,source_currency,VARCHAR
2,target_currency,VARCHAR
3,exchange_rate,DOUBLE
4,buy_rate,DOUBLE
5,sell_rate,DOUBLE
6,rate_source,VARCHAR


## 2. Row-count reconciliation + composite-key dedup

The `QUALIFY row_number() OVER (PARTITION BY date, source_currency, target_currency ORDER BY _ingested_at DESC) = 1` in `build_fx_rates_table` should collapse any duplicate rows sharing the same composite key, keeping the most recently ingested one.

In [6]:
bronze_total = q("SELECT count(*) AS n FROM bronze.daily_exchange_rates")["n"][0]
bronze_distinct_key = q("""
    SELECT count(*) AS n FROM (
        SELECT DISTINCT date, source_currency, target_currency FROM bronze.daily_exchange_rates
    )
""")["n"][0]
silver_total = q("SELECT count(*) AS n FROM silver.dim_fx_rates")["n"][0]
silver_distinct_key = q("""
    SELECT count(*) AS n FROM (
        SELECT DISTINCT rate_date, source_currency, target_currency FROM silver.dim_fx_rates
    )
""")["n"][0]

print(f"bronze.daily_exchange_rates: {bronze_total:,} rows, {bronze_distinct_key:,} distinct (date, source_currency, target_currency)")
print(f"silver.dim_fx_rates:         {silver_total:,} rows, {silver_distinct_key:,} distinct (rate_date, source_currency, target_currency)")
print()
print("Dictionary declares 3,000 rows -- matches" if bronze_total == 3000 else f"Dictionary declares 3,000 rows -- MISMATCH, Bronze has {bronze_total}")
print("Duplicate raw rows in Bronze (same composite key, multiple rows):", bronze_total - bronze_distinct_key)
print("OK -- Silver row count matches Bronze distinct composite-key count" if silver_total == bronze_distinct_key
      else "MISMATCH -- investigate: dedup may not be working as intended")

bronze.daily_exchange_rates: 13,164 rows, 13,164 distinct (date, source_currency, target_currency)
silver.dim_fx_rates:         13,164 rows, 13,164 distinct (rate_date, source_currency, target_currency)

Dictionary declares 3,000 rows -- MISMATCH, Bronze has 13164
Duplicate raw rows in Bronze (same composite key, multiple rows): 0
OK -- Silver row count matches Bronze distinct composite-key count


## 2.1 A closer look at any duplicated composite keys

If section 2 found duplicates, confirm Silver actually kept the most-recently-ingested row for each one (not an arbitrary one), same verification pattern used for `customers`/`products` dedup.

In [7]:
dupe_keys = q("""
    SELECT date, source_currency, target_currency, count(*) AS n
    FROM bronze.daily_exchange_rates
    GROUP BY 1, 2, 3
    HAVING count(*) > 1
    ORDER BY n DESC
""")
print(f"{len(dupe_keys)} composite keys with more than one raw Bronze row")
display(dupe_keys.head(20))

0 composite keys with more than one raw Bronze row


,date,source_currency,target_currency,n


In [8]:
# For one duplicated key (if any), confirm Silver kept the max(_ingested_at) row's values.
if len(dupe_keys) > 0:
    d, sc, tc = dupe_keys.iloc[0][["date", "source_currency", "target_currency"]]
    print(f"Bronze rows for ({d}, {sc}, {tc}):")
    display(q_full(f"""
        SELECT *, row_number() OVER (ORDER BY _ingested_at DESC) AS rn
        FROM bronze.daily_exchange_rates
        WHERE date = '{d}' AND source_currency = '{sc}' AND target_currency = '{tc}'
        ORDER BY _ingested_at DESC
    """))
    print("Silver's row for the same key:")
    display(q_full(f"""
        SELECT * FROM silver.dim_fx_rates
        WHERE rate_date = '{d}' AND source_currency = '{sc}' AND target_currency = '{tc}'
    """))
else:
    print("No duplicated composite keys -- nothing to verify.")

No duplicated composite keys -- nothing to verify.


## 3. NOT NULL audit -- Bronze raw vs. Silver typed

Dictionary NOT NULL: `date`, `source_currency`, `target_currency`, `exchange_rate` (the composite PK plus the rate itself). Nullable: `buy_rate`, `sell_rate`, `source`.

In [9]:
declared_not_null_bronze = ["date", "source_currency", "target_currency", "exchange_rate"]
declared_not_null_silver = ["rate_date", "source_currency", "target_currency", "exchange_rate"]

rows = []
for bcol, scol in zip(declared_not_null_bronze, declared_not_null_silver):
    bronze_nulls = q(f"""
        SELECT count(*) AS n FROM bronze.daily_exchange_rates
        WHERE {bcol} IS NULL OR trim(CAST({bcol} AS VARCHAR)) = ''
    """)["n"][0]
    silver_nulls = q(f"""
        SELECT count(*) AS n FROM silver.dim_fx_rates WHERE {scol} IS NULL
    """)["n"][0]
    rows.append({"bronze_column": bcol, "silver_column": scol, "declared": "NOT NULL",
                 "bronze_nulls": bronze_nulls, "silver_nulls": silver_nulls,
                 "cast_introduced_nulls": silver_nulls - bronze_nulls})

pd.DataFrame(rows)

,bronze_column,silver_column,declared,bronze_nulls,silver_nulls,cast_introduced_nulls
0,date,rate_date,NOT NULL,0,0,0
1,source_currency,source_currency,NOT NULL,0,0,0
2,target_currency,target_currency,NOT NULL,0,0,0
3,exchange_rate,exchange_rate,NOT NULL,0,0,0


## 4. Nullable columns (`buy_rate`/`sell_rate`/`source`->`rate_source`) -- population rate, Bronze vs. Silver

In [10]:
nullable_pairs = [("buy_rate", "buy_rate"), ("sell_rate", "sell_rate"), ("source", "rate_source")]
rows = []
total = q("SELECT count(*) AS n FROM bronze.daily_exchange_rates")["n"][0]
for bcol, scol in nullable_pairs:
    bronze_populated = q(f"""
        SELECT count(*) AS n FROM bronze.daily_exchange_rates
        WHERE {bcol} IS NOT NULL AND trim(CAST({bcol} AS VARCHAR)) <> ''
    """)["n"][0]
    silver_populated = q(f"""
        SELECT count(*) AS n FROM silver.dim_fx_rates WHERE {scol} IS NOT NULL
    """)["n"][0]
    rows.append({"bronze_column": bcol, "silver_column": scol, "total": total,
                 "bronze_populated": bronze_populated, "silver_populated": silver_populated,
                 "cast_dropped_values": bronze_populated - silver_populated})

pd.DataFrame(rows)

,bronze_column,silver_column,total,bronze_populated,silver_populated,cast_dropped_values
0,buy_rate,buy_rate,13164,13164,13164,0
1,sell_rate,sell_rate,13164,13164,13164,0
2,source,rate_source,13164,13164,13164,0


## 5. `source_currency`/`target_currency` -- domain check

Dictionary declares 4 currencies for the whole dataset: MXN, COP, ARS, USD. Not currently gated in `contracts.py`.

In [11]:
print("source_currency (Bronze):")
display(q("SELECT source_currency, count(*) AS n FROM bronze.daily_exchange_rates GROUP BY 1 ORDER BY n DESC"))
print("target_currency (Bronze):")
display(q("SELECT target_currency, count(*) AS n FROM bronze.daily_exchange_rates GROUP BY 1 ORDER BY n DESC"))
print("Distinct (source_currency, target_currency) pairs and their row counts:")
display(q("SELECT source_currency, target_currency, count(*) AS n FROM bronze.daily_exchange_rates GROUP BY 1, 2 ORDER BY n DESC"))

source_currency (Bronze):


,source_currency,n
0,MXN,3291
1,ARS,3291
2,USD,3291
3,COP,3291


target_currency (Bronze):


,target_currency,n
0,COP,3291
1,MXN,3291
2,ARS,3291
3,USD,3291


Distinct (source_currency, target_currency) pairs and their row counts:


,source_currency,target_currency,n
0,MXN,COP,1097
1,ARS,COP,1097
2,USD,MXN,1097
3,USD,ARS,1097
4,COP,MXN,1097
5,COP,ARS,1097
6,ARS,USD,1097
7,ARS,MXN,1097
8,USD,COP,1097
9,MXN,USD,1097


In [12]:
declared_currencies = {"MXN", "COP", "ARS", "USD"}
real_source = set(q("SELECT DISTINCT source_currency AS v FROM bronze.daily_exchange_rates")["v"])
real_target = set(q("SELECT DISTINCT target_currency AS v FROM bronze.daily_exchange_rates")["v"])
print("source_currency values NOT in the declared 4-currency set:", real_source - declared_currencies or "none -- full match")
print("target_currency values NOT in the declared 4-currency set:", real_target - declared_currencies or "none -- full match")
print()
print("Any row where source_currency == target_currency (a same-to-same 'rate')?")
display(q("SELECT * FROM bronze.daily_exchange_rates WHERE source_currency = target_currency LIMIT 10"))

source_currency values NOT in the declared 4-currency set: none -- full match
target_currency values NOT in the declared 4-currency set: none -- full match

Any row where source_currency == target_currency (a same-to-same 'rate')?


,date,source_currency,target_currency,exchange_rate,buy_rate,sell_rate,source,_source_file,_ingested_at,_source_table


## 6. `exchange_rate`/`buy_rate`/`sell_rate` -- cast correctness, range sanity, internal consistency

In [13]:
q("""
    SELECT
        typeof(exchange_rate) AS rate_type, typeof(buy_rate) AS buy_type, typeof(sell_rate) AS sell_type,
        min(exchange_rate) AS min_rate, max(exchange_rate) AS max_rate,
        count(*) FILTER (WHERE exchange_rate <= 0) AS non_positive_rates,
        count(*) FILTER (WHERE buy_rate IS NOT NULL AND buy_rate <= 0) AS non_positive_buy,
        count(*) FILTER (WHERE sell_rate IS NOT NULL AND sell_rate <= 0) AS non_positive_sell
    FROM silver.dim_fx_rates
    GROUP BY ALL
""")

,rate_type,buy_type,sell_type,min_rate,max_rate,non_positive_rates,non_positive_buy,non_positive_sell
0,DOUBLE,DOUBLE,DOUBLE,0.000245,4079.944006,0,0,0


In [14]:
# By currency pair: does the magnitude look like a real-world rate? (rough sanity bounds, not a hard check)
q("""
    SELECT source_currency, target_currency,
           count(*) AS n, round(min(exchange_rate), 4) AS min_rate, round(max(exchange_rate), 4) AS max_rate,
           round(avg(exchange_rate), 4) AS avg_rate
    FROM silver.dim_fx_rates
    GROUP BY 1, 2
    ORDER BY 1, 2
""")

,source_currency,target_currency,n,min_rate,max_rate,avg_rate
0,ARS,COP,1097,11.2010,11.6553,11.4351
1,ARS,MXN,1097,0.0476,0.0495,0.0486
2,ARS,USD,1097,0.0028,0.0029,0.0029
3,COP,ARS,1097,0.0858,0.0892,0.0874
4,COP,MXN,1097,0.0042,0.0043,0.0042
5,COP,USD,1097,0.0002,0.0003,0.0002
6,MXN,ARS,1097,20.1771,20.9988,20.5911
7,MXN,COP,1097,230.6027,239.9983,235.3037
8,MXN,USD,1097,0.0576,0.0600,0.0588
9,USD,ARS,1097,343.0101,356.9781,349.9909


In [15]:
# buy_rate <= exchange_rate <= sell_rate is the normal convention (bank buys low, sells high);
# check whether that holds, and how often buy_rate/sell_rate are populated together vs. exchange_rate alone.
q("""
    SELECT
        count(*) AS total,
        count(*) FILTER (WHERE buy_rate IS NOT NULL AND sell_rate IS NOT NULL) AS both_populated,
        count(*) FILTER (WHERE buy_rate IS NOT NULL AND sell_rate IS NOT NULL AND buy_rate > sell_rate) AS buy_gt_sell,
        count(*) FILTER (WHERE buy_rate IS NOT NULL AND buy_rate > exchange_rate) AS buy_gt_mid,
        count(*) FILTER (WHERE sell_rate IS NOT NULL AND sell_rate < exchange_rate) AS sell_lt_mid
    FROM silver.dim_fx_rates
""")

,total,both_populated,buy_gt_sell,buy_gt_mid,sell_lt_mid
0,13164,13164,0,0,0


## 7. `date`/`rate_date` -- cast correctness, coverage, window check

In [16]:
q("""
    SELECT
        typeof(rate_date) AS silver_type,
        count(*) FILTER (WHERE rate_date IS NULL) AS nulls,
        min(rate_date) AS earliest, max(rate_date) AS latest,
        count(DISTINCT rate_date) AS distinct_dates
    FROM silver.dim_fx_rates
    GROUP BY ALL
""")

,silver_type,nulls,earliest,latest,distinct_dates
0,DATE,0,2023-06-17,2026-06-17,1097


In [17]:
# Same window check as every other table.
q("""
    SELECT
        count(*) FILTER (WHERE rate_date < DATE '2023-06-17') AS before_window,
        count(*) FILTER (WHERE rate_date > DATE '2026-06-17') AS after_window
    FROM silver.dim_fx_rates
""")

,before_window,after_window
0,0,0


In [18]:
# Coverage check: for each currency pair, is there roughly one rate per day across the window
# (partition="daily" implies this), or are there gaps?
q("""
    SELECT source_currency, target_currency,
           min(rate_date) AS first_date, max(rate_date) AS last_date,
           count(DISTINCT rate_date) AS days_with_a_rate,
           (max(rate_date) - min(rate_date) + 1) AS calendar_days_in_span
    FROM silver.dim_fx_rates
    GROUP BY 1, 2
    ORDER BY 1, 2
""")

,source_currency,target_currency,first_date,last_date,days_with_a_rate,calendar_days_in_span
0,ARS,COP,2023-06-17,2026-06-17,1097,1097
1,ARS,MXN,2023-06-17,2026-06-17,1097,1097
2,ARS,USD,2023-06-17,2026-06-17,1097,1097
3,COP,ARS,2023-06-17,2026-06-17,1097,1097
4,COP,MXN,2023-06-17,2026-06-17,1097,1097
5,COP,USD,2023-06-17,2026-06-17,1097,1097
6,MXN,ARS,2023-06-17,2026-06-17,1097,1097
7,MXN,COP,2023-06-17,2026-06-17,1097,1097
8,MXN,USD,2023-06-17,2026-06-17,1097,1097
9,USD,ARS,2023-06-17,2026-06-17,1097,1097


## 8. `rate_source` (`source` in Bronze) -- free text spot check

In [19]:
q("SELECT rate_source, count(*) AS n FROM silver.dim_fx_rates GROUP BY 1 ORDER BY n DESC")

,rate_source,n
0,Bloomberg,3303
1,Reuters,3297
2,Internal,3293
3,Central Bank,3271


## 9. `silver.v_fx_latest_to_usd` -- the derived view other tables rely on for currency conversion

This view isn't a table `contracts.py` gates directly, but `products` (and possibly others) join against it for USD conversion when there's no natural as-of date. A bug here would silently propagate into every `amount_usd`-style estimate built from it.

In [20]:
print("The view's contents:")
display(q("SELECT * FROM silver.v_fx_latest_to_usd ORDER BY source_currency"))

The view's contents:


,source_currency,rate_to_usd,rate_date
0,ARS,0.002873,2026-06-17
1,COP,0.000248,2026-06-17
2,MXN,0.058641,2026-06-17


In [21]:
# Does it actually pick the latest rate_date per source_currency, restricted to target_currency='USD'?
# Reproduce it by hand and diff against the view.
reproduced = q("""
    SELECT source_currency, exchange_rate AS rate_to_usd, rate_date
    FROM silver.dim_fx_rates
    WHERE target_currency = 'USD'
    QUALIFY row_number() OVER (PARTITION BY source_currency ORDER BY rate_date DESC) = 1
    ORDER BY source_currency
""")
view_contents = q("SELECT * FROM silver.v_fx_latest_to_usd ORDER BY source_currency")
print("Reproduced by hand matches the view exactly:" , reproduced.equals(view_contents))
display(reproduced)

Reproduced by hand matches the view exactly: True


,source_currency,rate_to_usd,rate_date
0,ARS,0.002873,2026-06-17
1,COP,0.000248,2026-06-17
2,MXN,0.058641,2026-06-17


In [22]:
# Does every declared currency actually have a rate to USD at all? If MXN/COP/ARS -> USD is ever
# missing, v_fx_latest_to_usd silently omits that currency and any join against it for that
# currency returns nothing (not an error -- just a silent gap).
declared_currencies = {"MXN", "COP", "ARS", "USD"}
covered = set(q("SELECT DISTINCT source_currency AS v FROM silver.dim_fx_rates WHERE target_currency = 'USD'")["v"])
print("Currencies with NO rate to USD anywhere in the table:", declared_currencies - covered - {"USD"} or "none -- full coverage")

Currencies with NO rate to USD anywhere in the table: none -- full coverage


## Checklist

- [ ] Column inventory (1): rename mapping (`date`->`rate_date`, `source`->`rate_source`) correctly applied, types correct
- [ ] Row reconciliation (2) + dedup (2.1): 3,000 raw rows, composite-key dedup keeps the most-recently-ingested row
- [ ] NOT NULL audit (3): `date`/`source_currency`/`target_currency`/`exchange_rate` fully populated, no cast-introduced nulls
- [ ] Nullable columns (4): `buy_rate`/`sell_rate`/`source` population rates sane, no cast silently dropped a value
- [ ] Currency domain (5): `source_currency`/`target_currency` only ever MXN/COP/ARS/USD, no same-to-same rows
- [ ] Rate sanity (6): no non-positive rates, `buy_rate <= exchange_rate <= sell_rate` holds where populated
- [ ] Date coverage (7): no window violations, no unexpected gaps in daily coverage per currency pair
- [ ] `rate_source` (8): values make sense
- [ ] `v_fx_latest_to_usd` (9): reproduces exactly by hand, every currency has a path to USD

In [23]:
con.close()
print("closed")

closed
